# Assignment 02: Fetching Occurrence Records With pygbif

## BIO597 Spatial Analysis of Biodiversity

This assignment gives you more practice with the workflow from Lab 02.

You will use `pygbif` to search GBIF for several snake species, download a small number of occurrence records, convert those records to `GeoDataFrame` objects, and make simple maps and summaries.

Some cells are partly filled in. You should fill in the missing pieces and run the notebook from top to bottom.

Remember, learning to code is often about learning how to strategically copy, paste, and modify. Look back at Lab 02 when you need a model.


## Species for this assignment

Use these species:

* *Storeria dekayi*
* *Agkistrodon contortrix*
* *Pantherophis guttatus*
* one additional snake species of your choice

For each species, we will ask GBIF for georeferenced records so the results can be mapped.


## 1. Import packages

Import the same packages used in Lab 02.

`species` is used for taxonomic name searches. `occ` is used for occurrence record searches.


In [23]:
from pygbif import species
from pygbif import occurrences as occ

import pandas as pd
import geopandas as gpd

## 2. Search for possible name matches

Use `species.name_suggest()` to search for *Storeria dekayi*.

Fill in the species name. Then inspect the first result.


In [27]:
dekayi_suggestions = species.name_suggest(q="Storeria dekayi")
# dekayi_suggestions will have a list of dictionaries
# select the first element of this list here and save it as a new variable called `dekayi_match`
dekayi_match = dekayi_suggestions[0]
print(dekayi_match)

{'key': 9056579, 'nameKey': 10782717, 'kingdom': 'Animalia', 'phylum': 'Chordata', 'family': 'Colubridae', 'genus': 'Storeria', 'species': 'Storeria dekayi', 'kingdomKey': 1, 'phylumKey': 44, 'classKey': 11592253, 'familyKey': 6172, 'genusKey': 9213370, 'speciesKey': 9056579, 'parent': 'Storeria', 'parentKey': 9213370, 'nubKey': 9056579, 'scientificName': 'Storeria dekayi (Holbrook, 1839)', 'canonicalName': 'Storeria dekayi', 'rank': 'SPECIES', 'status': 'ACCEPTED', 'synonym': False, 'higherClassificationMap': {'1': 'Animalia', '44': 'Chordata', '11592253': 'Squamata', '6172': 'Colubridae', '9213370': 'Storeria'}, 'class': 'Squamata'}


## 3. Save the taxon key

Pull the `speciesKey` out of the match result and save it as `dekayi_key`.


In [28]:
dekayi_key = dekayi_match["speciesKey"]
dekayi_key

9056579

## 4. Count records before downloading

Use `occ.count()` to count georeferenced GBIF records for *Storeria dekayi*.

This count tells you how many records GBIF has that match your search, not how many you will download in this assignment.


In [29]:
dekayi_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=True,
)

dekayi_count

49473

## 5. Determine how many _total_ records there are for S. dekayi

## 5. Determine how many _total_ records there are for S. dekayi

The `isGeoreferenced` parameter determines whether occurrences with latlongs are returned.
Make a copy of the call to `occ.count()` as above, but change the `True` to `False`, which
will return only occurrences **without** latlongs. Capture the results in a new variable 
called `dekayi_nolatlong_count` and then add this to `dekayi_count` from the previous cell
to get the total number of records.

In [86]:
# How many total S. Dekayi records are there?
dekayi_nolatlong_count = occ.count(
    taxonKey = dekayi_key,
    isGeoreferenced = False,
)
total_dekayi_count = (dekayi_count + dekayi_nolatlong_count) 
total_dekayi_count

56546

## 6. Fetch up to 100 records

Use `occ.search()` to fetch occurrence records for *Storeria dekayi*.

Keep `limit=100`. Do not request more than 100 records for a species in this assignment.


In [31]:
dekayi_records = occ.search(
    speciesKey= dekayi_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

dekayi_records.keys()

dict_keys(['offset', 'limit', 'endOfRecords', 'count', 'results', 'facets'])

## 7. Turn the records into a table

The occurrence records are stored in the "results" key of the `dekayi_records` dictionary. Convert that list of records to a pandas DataFrame.


In [35]:
dekayi_df = pd.DataFrame(dekayi_records["results"])
dekayi_df.head()
# is there a way to see all 97 column names?
for col in dekayi_df:
    print(col)

key
datasetKey
publishingOrgKey
datasetCategory
installationKey
hostingOrganizationKey
publishingCountry
protocol
lastCrawled
lastParsed
crawlId
extensions
basisOfRecord
occurrenceStatus
classifications
taxonKey
kingdomKey
phylumKey
classKey
familyKey
genusKey
speciesKey
acceptedTaxonKey
scientificName
scientificNameAuthorship
acceptedScientificName
kingdom
phylum
family
genus
species
genericName
specificEpithet
taxonRank
taxonomicStatus
iucnRedListCategory
dateIdentified
decimalLatitude
decimalLongitude
coordinateUncertaintyInMeters
continent
stateProvince
gadm
year
month
day
eventDate
startDayOfYear
endDayOfYear
issues
modified
lastInterpreted
references
license
isSequenced
identifiers
media
facts
relations
isInCluster
datasetName
recordedBy
identifiedBy
dnaSequenceID
nucleotideSequence
geodeticDatum
class
countryCode
recordedByIDs
identifiedByIDs
gbifRegion
country
publishedByGbifRegion
rightsHolder
identifier
http://unknown.org/nick
verbatimEventDate
http://unknown.org/crawl_attemp

## 8. Keep a small set of useful columns

Keep the columns needed for a map and a few simple summaries.

Fill in the latitude column name.


In [33]:
dekayi_small = dekayi_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

dekayi_small.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US


## 9. Convert the table to a GeoDataFrame

Use the longitude and latitude columns to create point geometry. This is the same idea as Lab 01 and Lab 02.


In [36]:
dekayi_gdf = gpd.GeoDataFrame(
    dekayi_small,
    geometry=gpd.points_from_xy(dekayi_small["decimalLongitude"], dekayi_small["decimalLatitude"]),
    crs="EPSG:4326",
)

dekayi_gdf["Species"] = "Storeria dekayi"
dekayi_gdf.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode,geometry,Species
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US,POINT (-97.12243 33.24141),Storeria dekayi
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US,POINT (-92.90881 34.6199),Storeria dekayi
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US,POINT (-86.7215 33.45804),Storeria dekayi
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US,POINT (-80.7443 35.17294),Storeria dekayi
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US,POINT (-97.50869 35.239),Storeria dekayi


## 10. Map *Storeria dekayi*

Make an interactive map of the *Storeria dekayi* records.


In [37]:
dekayi_gdf.explore()

## 11. Repeat the workflow for *Agkistrodon contortrix*

Now repeat the same steps for eastern copperhead, *Agkistrodon contortrix*.

This cell should get the name suggestions and save the speciesKey.


In [38]:
contortrix_suggestions = species.name_suggest(q="Agkistrodon contortrix")

# Select the first element from the `contortrix_suggestions` list
contortrix_match = contortrix_suggestions[0]

# Get the `speciesKey`
contortrix_key = contortrix_match["speciesKey"]

contortrix_key

9215881

## 13. Fetch and map *Agkistrodon contortrix*

Write code to fetch up to 100 records with coordinates, convert them to a table, convert that table to a GeoDataFrame, add a `Species` column, and map the result.

Use the same variable names shown in the comments. In `occ.search()`, use `hasCoordinate=True` and `hasGeospatialIssue=False`.


In [84]:
contortrix_count = occ.count(
    taxonKey=contortrix_key,
    isGeoreferenced=True,
)
# How many total A. contortrix records are there?
contortrix_nolatlong_count = occ.count(
    taxonKey = contortrix_key,
    isGeoreferenced = False,
)
print(contortrix_nolatlong_count)
print(contortrix_count)
contortrix_total = print(contortrix_nolatlong_count + contortrix_count)
# can I do it this way?
contortrix_total = occ.count(
    taxonKey = contortrix_key,
)

5812
23303
29115


In [85]:
print(contortrix_total)

29115


In [58]:


# Create contortrix_records with occ.search().
contortrix_records = occ.search(
    speciesKey= contortrix_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)


total_dekayi_count = print(dekayi_count + dekayi_nolatlong_count) 

# Create contortrix_df from contortrix_records["results"].
contortrix_df = pd.DataFrame(contortrix_records["results"])
contortrix_df.head()
# Create contortrix_small with the columns you want to keep.
contortrix_small = contortrix_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

# Create contortrix_gdf with gpd.GeoDataFrame().
contortrix_gdf = gpd.GeoDataFrame(contortrix_small,
                                  geometry=gpd.points_from_xy(contortrix_small["decimalLongitude"], 
                                                              contortrix_small["decimalLatitude"]), 
                                    crs="EPSG:4326",
)
# Add a Species column with the name Agkistrodon contortrix.
contortrix_gdf["Species"] = "Agkistrodon contortrix"
# Map contortrix_gdf with .explore().
contortrix_gdf.explore()

56546


## 14. Repeat the workflow for *Pantherophis guttatus*

This time you will do a little more on your own.

First, use `species.name_suggest()` to get the `speciesKey` for *Pantherophis guttatus*.


In [48]:
# Write your code here.
guttatus_suggestions = species.name_suggest(q="Pantherophis guttatus")
guttatus_match = guttatus_suggestions[0]

# Get the `speciesKey`
guttatus_key = guttatus_match["speciesKey"]

guttatus_key

2455615

## 16. Fetch and map *Pantherophis guttatus*

Fetch up to 100 georeferenced records and convert them to a GeoDataFrame.

Keep the cell simple. It is fine to copy and modify code from earlier cells.


In [87]:
guttatus_total = occ.count(
    taxonKey = guttatus_key,
)
guttatus_total

13214

In [49]:
# Write your code here.
guttatus_records = occ.search(
    speciesKey= guttatus_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

guttatus_df = pd.DataFrame(guttatus_records["results"])
guttatus_df.head()
guttatus_small = guttatus_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]
guttatus_gdf = gpd.GeoDataFrame(guttatus_small,
                                  geometry=gpd.points_from_xy(guttatus_small["decimalLongitude"], 
                                                              guttatus_small["decimalLatitude"]), 
                                    crs="EPSG:4326",
)

guttatus_gdf["Species"] = "Pantherophis guttatus"

guttatus_gdf.explore()

## 17. Combine the three GeoDataFrames

Use `pd.concat()` to combine your three species GeoDataFrames.

Then inspect the first few rows.


In [50]:
gbif_snakes = pd.concat([guttatus_gdf, contortrix_gdf, dekayi_gdf])

gbif_snakes.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode,geometry,Species
0,5938135213,"Pantherophis guttatus (Linnaeus, 1766)",-81.394973,30.135123,2026,HUMAN_OBSERVATION,US,POINT (-81.39497 30.13512),Pantherophis guttatus
1,5938250009,"Pantherophis guttatus (Linnaeus, 1766)",-80.278505,25.329077,2026,HUMAN_OBSERVATION,US,POINT (-80.2785 25.32908),Pantherophis guttatus
2,5938256663,"Pantherophis guttatus (Linnaeus, 1766)",-80.452592,27.789722,2026,HUMAN_OBSERVATION,US,POINT (-80.45259 27.78972),Pantherophis guttatus
3,5938387083,"Pantherophis guttatus (Linnaeus, 1766)",-88.015381,30.231975,2026,HUMAN_OBSERVATION,US,POINT (-88.01538 30.23198),Pantherophis guttatus
4,5938619022,"Pantherophis guttatus (Linnaeus, 1766)",-81.797488,26.245595,2026,HUMAN_OBSERVATION,US,POINT (-81.79749 26.2456),Pantherophis guttatus


## 18. Map all three species together

Use `.explore()` and color by `Species` so you can compare the three species on one map.


In [51]:
gbif_snakes.explore(column="Species", cmap="rainbow")

## 19. Count records by species

Use `groupby()` to count how many records you downloaded for each species.


In [52]:
gbif_snakes.groupby("Species").size()

Species
Agkistrodon contortrix    100
Pantherophis guttatus     100
Storeria dekayi           100
dtype: int64

## 20. Count records by basis of record

The `countryCode` field describes the general type of occurrence record.

Use `groupby()` to count records by `countryCode`.


In [53]:
gbif_snakes.groupby("countryCode").size()

countryCode
MX      2
US    298
dtype: int64

## 21. Choose one additional species

Choose one additional snake species and repeat the workflow.

Your species does not have to be in the local `EasternSnakes` CSV files. It only needs to be a snake species that GBIF can find.

Your code should:

* use `species.name_suggest()`
* save the taxon key
* count georeferenced records
* fetch no more than 100 records with `occ.search()`
* convert the records to a `GeoDataFrame`
* map the records


In [88]:
# Write your code here.
vernalis_suggestions = species.name_suggest(q="Opheodrys vernalis")
vernalis_match = vernalis_suggestions[0]

# Get the `speciesKey`
vernalis_key = vernalis_match["speciesKey"]

vernalis_key

vernalis_total = occ.count(
    taxonKey = vernalis_key,
)

vernalis_records = occ.search(
    speciesKey= vernalis_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

vernalis_df = pd.DataFrame(vernalis_records["results"])
vernalis_df.head()
vernalis_small = vernalis_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]
vernalis_gdf = gpd.GeoDataFrame(vernalis_small,
                                  geometry=gpd.points_from_xy(vernalis_small["decimalLongitude"], 
                                                              vernalis_small["decimalLatitude"]), 
                                    crs="EPSG:4326",
)

vernalis_gdf["Species"] = "Opheodrys vernalis"

gbif_snakes_vern = pd.concat([guttatus_gdf, contortrix_gdf, dekayi_gdf, vernalis_gdf])
gbif_snakes_vern.head()

gbif_snakes_vern.explore(column="Species", cmap="rainbow")

In [91]:
# list all species counts
print( contortrix_total, vernalis_total, guttatus_total, total_dekayi_count,) 




29115 9323 13214 56546


In [93]:
acontortrix_csv = "../labs/EasternSnakes/Acontortrix_coords.csv"
acontortrix_df = gpd.read_file(acontortrix_csv)
acontortrix_gdf = gpd.GeoDataFrame(
    acontortrix_df,
    geometry=gpd.points_from_xy(acontortrix_df["Longitude"], acontortrix_df["Latitude"]),
    crs="EPSG:4326",
)

acontortrix_gdf["Species"] = "Agkistrodon contortrix"

pguttatus_csv = "../labs/EasternSnakes/Pguttatus_coords.csv"
pguttatus_df = gpd.read_file(pguttatus_csv)
pguttatus_gdf = gpd.GeoDataFrame(
    pguttatus_df,
    geometry=gpd.points_from_xy(pguttatus_df["Longitude"], pguttatus_df["Latitude"]),
    crs="EPSG:4326",
)

pguttatus_gdf["Species"] = "Punctatus guttatus"

sdekayi_csv = "../labs/EasternSnakes/Sdekayi_coords.csv"
sdekayi_df = gpd.read_file(sdekayi_csv)
sdekayi_gdf = gpd.GeoDataFrame(
    sdekayi_df,
    geometry=gpd.points_from_xy(sdekayi_df["Longitude"], sdekayi_df["Latitude"]),
    crs="EPSG:4326",
)

sdekayi_gdf["Species"] = "Storeria dekayi"

In [97]:
csv_snakes = pd.concat([sdekayi_gdf, pguttatus_gdf, acontortrix_gdf])

In [103]:
csv_snakes.explore(column = "Species", col = "Rainbow") 


In [104]:
gbif_snakes.explore(column = "Species", col = "Rainbow")

## 22. Written reflection

Answer these questions after running your code.

**Question 1:** Which of your species had the most GBIF records available?

**Your answer:** S. dekayi has the most GBIF records available (56,546)

**Question 2:** Did the GBIF points look similar to the local CSV points from Assignment 01? Why might GBIF records look different?

**Your answer:** The CSV points look like they were preprocessed, probably thinned to reduce spatial autocorrelation. The GBIF records are more clumped in appearance; that could be because points in the database are listed together by source, so the first 100 records are more likely to be from the same sampling area, depending on the place where records came from.  

**Question 3:** What is one reason it is useful to count records before downloading or mapping them?

**Your answer:** It is good to have an idea of what percentage of the records you are obtaining. This could affect the spread of the data you are downloading, where the entire range of the species is not covered. If the data looks weird when mapped it may be that the sample is not representative of the population. 


## 23. Submit your work

Before submitting, make sure you have run the notebook from top to bottom and answered the written questions.

Commit and push your completed notebook to your class GitHub repository.

Open a terminal window and run these commands to add, commit, and push your notebook:

```
# Go to the labs directory in the course repo
cd ~/BIO597-SpatialBiodiversity/docs/assignments

# Add your changed lab
git add Assignment-02-pygbif.ipynb
git commit -m 'Finished Assignment 02'
git push
```